# P0 parity check ? DenseNet121 EXP-024/025 (Colab)

Run this notebook in the original Linux/GPU environment with TensorFlow 2.21.0 and Keras 3.13.2. It reads the frozen EXP-024 deployment model, EXP-025 saved Holdout predictions, and Holdout (5) ZIP. It **does not train, evaluate the full Test/Holdout, relabel, or select a new seed**. It checks five saved predictions and stops when confidence parity exceeds 1e-4. Grad-CAM must not run until this gate passes.


In [ ]:
import sys, json, hashlib, zipfile, io
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from google.colab import drive

drive.mount('/content/drive')
print('Python:', sys.version)
print('TensorFlow:', tf.__version__, 'Keras:', keras.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
if tf.__version__ != '2.21.0' or keras.__version__ != '3.13.2':
    raise RuntimeError('Use the original TensorFlow 2.21.0 / Keras 3.13.2 runtime before parity testing')
if not tf.config.list_physical_devices('GPU'):
    raise RuntimeError('Use a Colab GPU runtime matching the original EXP-025 run')


In [ ]:
# Adjust only these paths to the mounted project shortcut. Do not alter model recipe.
PROJECT = Path('/content/drive/MyDrive/Cashew_Leaf_model_result')
RESULTS = PROJECT / 'training_results' / 'current_dataset'
EXP024 = RESULTS / 'EXP-DENSENET121-TRANSFER-5SEEDS-024-256px-f9fd5d5f'
EXP025 = RESULTS / 'EXP-DENSENET121-TRANSFER-5SEEDS-025-256px'
HOLDOUT_ZIP = PROJECT / 'Dataset' / 'RL_Cashew_holdout (5).zip'
MODEL = EXP024 / 'model_package' / 'deployment_model.keras'
LOCK = EXP024 / 'validation_selection_lock.json'
CONFIG = EXP024 / 'experiment_config.json'
PREDICTIONS = EXP025 / 'real_holdout' / 'seed_42' / 'predictions.csv'
for p in (HOLDOUT_ZIP, MODEL, LOCK, CONFIG, PREDICTIONS):
    if not p.is_file(): raise FileNotFoundError(p)
print('EXP-024:', EXP024)
print('EXP-025:', EXP025)


In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1 << 20), b''): h.update(block)
    return h.hexdigest()

lock = json.loads(LOCK.read_text(encoding='utf-8'))
config = json.loads(CONFIG.read_text(encoding='utf-8'))
assert lock['deployment_seed'] == 42
assert config['experiment_id'] == 'EXP-DENSENET121-TRANSFER-5SEEDS-024-256px-f9fd5d5f'
assert config['dataset']['class_names'] == ['anthracnose','healthy','leaf_miner','red_rust','not_cashew_leaf']
assert sha256(CONFIG) == lock['config_sha256']
assert sha256(HOLDOUT_ZIP) == '8d5c3758b5869c3ae19238fa8d78e53ad15bb40544f4c70f632602896d80045f'
# deployment_model.keras and best.weights.h5 are different formats and have separate hashes.
assert sha256(MODEL) == '8bcd23bcadfa83c81264feceba89eef40d9b5518b9497d4e51c50fe6a6d8dd0a'
WEIGHTS = EXP024 / 'seed_42' / 'best.weights.h5'
assert sha256(WEIGHTS) == lock['selected_checkpoint_sha256']['42']
print('Source and checkpoint integrity: PASS')


In [ ]:
CLASS_NAMES = ['anthracnose','healthy','leaf_miner','red_rust','not_cashew_leaf']
paths = ['anthracnose/real_anthracnose_051.jpg',
         'healthy/real_healthy_027.jpg',
         'leaf_miner/real_leaf_miner_131.jpg',
         'red_rust/real_red_rust_031.jpg',
         'healthy/real_healthy_022.jpg']
predictions = pd.read_csv(PREDICTIONS).set_index('relative_path')
model = keras.models.load_model(MODEL, compile=False)
rows = []
with zipfile.ZipFile(HOLDOUT_ZIP) as archive:
    names = {name.split('RL_Cashew_holdout/', 1)[-1]:name for name in archive.namelist()
             if name.lower().endswith(('.jpg','.jpeg','.png'))}
    for path in paths:
        raw = archive.read(names[path])
        image = tf.io.decode_image(raw, channels=3, expand_animations=False)
        image.set_shape([None,None,3])
        image = tf.cast(tf.image.resize(image, [256,256], method='bilinear'), tf.float32)
        prob = model(image[None,...], training=False).numpy()[0]
        pred = CLASS_NAMES[int(np.argmax(prob))]
        log = predictions.loc[path]
        confidence = float(np.max(prob))
        diff = abs(confidence-float(log['confidence']))
        rows.append({'relative_path':path,'logged_label':log['predicted_class'],
                     'model_label':pred,'logged_confidence':float(log['confidence']),
                     'model_confidence':confidence,'confidence_abs_diff':diff,
                     'pass':pred==log['predicted_class'] and diff<=1e-4})
result = pd.DataFrame(rows)
display(result)
if not result['pass'].all():
    raise RuntimeError('PARITY FAILED ? do not generate Grad-CAM. Inspect versions, image decode, resize and model source.')
print('PARITY PASS ? seed-42 Grad-CAM may proceed from the verified model')


If the original `EXP024` directory is unavailable but the FULL ZIP is mounted, extract only the exact source folder to a new Colab working directory first. Do not overwrite the original result folder. A failed parity check is a diagnostic result; retain the displayed table and stop before CAM.